# EasyRAG QueryResult To Answer

## What you will do

- build a tiny zero-key workspace with deterministic retrieval stubs
- run one grounded retrieval query and inspect the raw citations
- select a smaller answer-ready evidence set
- pack that evidence into a context block and a prompt
- produce a simple citation-aware answer with a local teaching helper

## Why this notebook uses a local answer helper

The goal here is not to hide generation behind another provider call. The goal is to make context selection, prompt construction, and answer synthesis visible. The answer helper is therefore notebook-local and deterministic.

## Source code anchors

- `easyrag.rag.types.QueryResult`
- `easyrag.rag.generation.selection.select_answer_citations`
- `easyrag.rag.generation.packing.build_context_block`
- `easyrag.rag.generation.prompting.build_generation_prompt`


## Step 1: Import the public retrieval API and runtime helper

We only need the public EasyRAG surface plus a few Python standard-library utilities. The generation-specific logic in this notebook will be implemented as local helpers layered on top of `QueryResult`.


In [ ]:
import json
import re
import tempfile

from easyrag import EasyRAG, QueryParam
from easyrag.support.async_utils import run_sync


This cell should not produce meaningful output. It just establishes the public API surface and the standard-library tools we need for prompt assembly.


## Step 2: Define deterministic retrieval stubs

These stubs keep the retrieval side reproducible. The embedding helper creates simple dense vectors from domain keywords, while the query helper gives us stable rewrite and MQE behavior without requiring an external model.


In [ ]:
# ruff: noqa: E402,F401,F811
import sys
from pathlib import Path

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "easyrag").exists():
        REPO_ROOT = candidate.resolve()
        if str(REPO_ROOT) not in sys.path:
            sys.path.insert(0, str(REPO_ROOT))
        break
else:
    raise RuntimeError("Could not locate the EasyRAG repository root.")

import json
import os
import tempfile
import time
from pathlib import Path
from pprint import pprint
from unittest import mock

from easyrag.rag import AnswerParam, EasyRAG, EvalCase, QueryParam
from easyrag.support.async_utils import run_sync
from notebooks._utils import (
    managed_demo_rag,
    print_json as _print_json,
    production_backends_ready,
    provider_ready as can_use_openai_compatible_models,
    skip_message,
    stub_embedding as _stub_embedding,
    stub_query_model as _stub_query_model,
    stub_reranker as _stub_reranker,
)


This cell only defines retrieval behavior, so there should be no runtime output yet. The next step adds notebook-local helpers for evidence selection and answer composition.


## Step 3: Define local evidence-selection and answer helpers

This is the generation-focused part of the notebook. None of these helpers are part of the current EasyRAG public API. They are teaching utilities that show what usually happens after a grounded `QueryResult` has already been returned.


In [ ]:
def _normalize_snippet(text: str) -> str:
    cleaned = str(text).replace("#", " ")
    return " ".join(cleaned.split())


def _pick_answer_citations(
    citations: list[dict[str, str]],
    *,
    max_items: int = 3,
    max_chars: int = 360,
) -> list[dict[str, str]]:
    selected: list[dict[str, str]] = []
    budget = 0
    for citation in citations:
        cleaned = dict(citation)
        cleaned["snippet"] = _normalize_snippet(cleaned.get("snippet", ""))[:220]
        snippet_chars = len(cleaned["snippet"])
        if selected and (len(selected) >= max_items or budget + snippet_chars > max_chars):
            break
        selected.append(cleaned)
        budget += snippet_chars
    return selected


def _citation_flow_rank(citation: dict[str, str]) -> tuple[int, str]:
    location = citation.get("location", "").lower()
    if "ingress" in location:
        return (0, location)
    if "validation" in location:
        return (1, location)
    if "storage" in location:
        return (2, location)
    return (99, location)


def _order_citations_for_answer(citations: list[dict[str, str]]) -> list[dict[str, str]]:
    return sorted(citations, key=_citation_flow_rank)


def _build_context_block(citations: list[dict[str, str]]) -> str:
    blocks = []
    for index, citation in enumerate(citations, start=1):
        blocks.append(
            f"[{index}] {citation['title']} ({citation['location']})\n"
            f"{_normalize_snippet(citation['snippet'])}"
        )
    return "\n\n".join(blocks)


def _build_generation_prompt(question: str, citations: list[dict[str, str]]) -> str:
    context_block = _build_context_block(citations)
    return "\n".join(
        [
            "You answer questions using only the retrieved evidence.",
            "If the evidence is incomplete, say so clearly.",
            "Use citation markers like [1] and [2] when you make a claim.",
            "",
            f"Question: {question}",
            "",
            "Retrieved evidence:",
            context_block,
            "",
            "Write a short grounded answer.",
        ]
    )


def _split_sentences(text: str) -> list[str]:
    parts = re.split(r"(?<=[.!?])\s+", _normalize_snippet(text))
    return [part.strip() for part in parts if part.strip()]


def _stub_answer_model(question: str, citations: list[dict[str, str]]) -> str:
    keywords = [token.lower() for token in re.findall(r"[A-Za-z]+", question) if len(token) > 3]
    ranked: list[tuple[int, int, str, dict[str, str]]] = []
    for index, citation in enumerate(citations, start=1):
        for sentence in _split_sentences(citation.get("snippet", "")):
            lowered = sentence.lower()
            score = sum(lowered.count(keyword) for keyword in keywords)
            if score > 0:
                ranked.append((score, index, sentence, citation))

    ranked.sort(key=lambda item: (-item[0], item[1], item[2]))

    selected_sentences: list[tuple[int, str, dict[str, str]]] = []
    seen_sentences: set[str] = set()
    for _, index, sentence, citation in ranked:
        if sentence in seen_sentences:
            continue
        selected_sentences.append((index, sentence, citation))
        seen_sentences.add(sentence)
        if len(selected_sentences) == 3:
            break

    if not selected_sentences and citations:
        selected_sentences.append((1, _normalize_snippet(citations[0]["snippet"]), citations[0]))

    selected_sentences.sort(key=lambda item: item[0])

    answer_body = " ".join(
        sentence if sentence.endswith((".", "!", "?")) else f"{sentence}."
        for _, sentence, _ in selected_sentences
    )
    used_indices = []
    references = []
    for index, _, citation in selected_sentences:
        if index in used_indices:
            continue
        used_indices.append(index)
        references.append(f"[{index}] {citation['title']} - {citation['location']}")

    if not answer_body:
        answer_body = "I cannot answer the question from the retrieved evidence alone."

    return answer_body + "\n\nSupporting citations:\n" + "\n".join(references)


This cell still should not print anything. The important change is conceptual: we now have a visible generation layer built on top of citations rather than hidden behind another black-box call.


## Step 4: Create a tiny corpus with an obvious answer path

The corpus is deliberately small and process-oriented. We want the retrieval output to contain a traceable flow from ingress to validation to durable storage so that the later answer synthesis step can stay grounded.


In [ ]:
demo_texts = [
    "# Ingress\nGateway receives billing requests from clients and forwards them to Policy Engine for validation.\n",
    "# Validation\nPolicy Engine checks request rules and passes approved billing work to Billing Service.\n",
    "# Storage\nBilling Service writes invoice state to Ledger Store and emits an event for Reporting Worker.\n",
]

demo_ids = ["doc::ingress", "doc::validation", "doc::storage"]
demo_paths = [
    "architecture/ingress.md",
    "architecture/validation.md",
    "architecture/storage.md",
]

print(json.dumps({"ids": demo_ids, "paths": demo_paths}, indent=2))


You should see three logical document IDs and paths. These logical paths later show up in citations, which makes the final answer easier to audit.


## Step 5: Create and initialize a temporary workspace

We will inject the retrieval stubs into a normal `EasyRAG` instance so the indexing and retrieval flow is still real, even though the model-facing pieces are deterministic.


In [ ]:
temp_dir = tempfile.TemporaryDirectory()
rag = EasyRAG(
    working_dir=temp_dir.name,
    workspace="generation-basics",
    embedding_func=_stub_embedding,
    query_model_func=_stub_query_model,
)
run_sync(rag.initialize_storages())

print(json.dumps({"working_dir": temp_dir.name, "workspace": rag.workspace}, indent=2))


You should see a temporary working directory and the `generation-basics` workspace name. At this point the workspace exists, but it does not contain knowledge yet.


## Step 6: Insert the corpus and run one grounded retrieval query

This is the retrieval half of the story. We index the tiny corpus, ask one question, and keep the full `QueryResult` so that generation can start from inspectable evidence instead of hidden model state.


In [ ]:
insert_stats = run_sync(rag.ainsert(demo_texts, ids=demo_ids, file_paths=demo_paths))
question = "How does a billing request reach the ledger store?"
result = run_sync(
    rag.aquery(
        question,
        QueryParam(mode="hybrid", chunk_top_k=4, rewrite_enabled=True, mqe_enabled=True),
    )
)

print(
    json.dumps(
        {
            "insert_stats": insert_stats,
            "mode": result.mode,
            "citation_count": len(result.citations),
            "metadata": {
                key: result.metadata[key]
                for key in ("rewritten_query", "expanded_queries", "retrieval_queries", "vector_backend")
            },
        },
        indent=2,
    )
)


A successful run should show nonzero insert counts, a nonzero citation count, and retrieval metadata such as the rewritten query and expanded queries. That confirms we are starting generation from a real `QueryResult`.


## Step 7: Inspect the raw grounded evidence

Before selecting or packing anything, look at the citations exactly as retrieval returned them. This is the easiest way to keep the later answer honest.


In [ ]:
for index, citation in enumerate(result.citations, start=1):
    print(f"[{index}] {citation['title']} | {citation['location']}")
    print(_normalize_snippet(citation["snippet"]))
    print()


You should see the ingress, validation, and storage evidence in a readable form. This is the raw material generation has to work with. Nothing has been packed or rewritten yet.


## Step 8: Select a smaller answer-ready citation set

Generation normally performs a second round of trimming after retrieval. The retrieval stage may surface several useful citations, but the answer stage still needs to decide how much evidence can fit cleanly into one prompt.


In [ ]:
selected_citations = _order_citations_for_answer(
    _pick_answer_citations(result.citations, max_items=3, max_chars=360)
)
raw_chars = sum(len(_normalize_snippet(citation["snippet"])) for citation in result.citations)
selected_chars = sum(len(citation["snippet"]) for citation in selected_citations)

print(
    json.dumps(
        {
            "retrieved_citations": len(result.citations),
            "selected_citations": len(selected_citations),
            "raw_chars": raw_chars,
            "selected_chars": selected_chars,
            "selected_locations": [citation["location"] for citation in selected_citations],
        },
        indent=2,
    )
)


The main observation here is that generation often works with a smaller evidence packet than retrieval originally returned. That is the start of context packing, not a loss of grounding.


## Step 9: Pack the selected citations into a context block

This step makes the evidence portable. We keep titles, logical paths, and snippets together so the prompt can preserve source boundaries instead of flattening everything into anonymous text.


In [ ]:
context_block = _build_context_block(selected_citations)
print(context_block)


You should see a compact evidence block with stable citation markers. This is a much better generation input than an unstructured pile of raw chunks.


## Step 10: Build a simple grounded prompt

The prompt joins together instructions, the user question, and the packed evidence block. In a larger system this would likely be handled by a dedicated prompt builder, but here we keep it visible and local.


In [ ]:
prompt = _build_generation_prompt(question, selected_citations)
print(prompt)


The prompt should now make the answer contract explicit: use only retrieved evidence, cite claims, and stay short. This notebook-level prompt builder is intentionally simple because the teaching goal is transparency.


## Step 11: Synthesize a citation-aware answer

The final step turns the packed evidence into a short answer. The helper below is deterministic on purpose, so you can focus on the handoff structure rather than model variability.


In [ ]:
answer = _stub_answer_model(question, selected_citations)
print(answer)


You should see a short answer assembled from retrieved evidence plus a supporting citation list. The important lesson is not that this helper is sophisticated. The important lesson is that the answer is downstream of inspectable evidence.


## Step 12: Clean up the temporary workspace

As in the earlier fundamentals notebooks, we finalize storages and remove the temporary workspace when the teaching run is complete.


In [ ]:
run_sync(rag.finalize_storages())
temp_dir.cleanup()
print("Workspace finalized and temporary directory removed.")


['After cleanup, the full retrieval-to-generation handoff is complete.\n', '\n', '## What this notebook taught\n', '\n', '- retrieval returns grounded evidence, not the final answer\n', '- generation usually performs a second evidence-selection step\n', '- context packing preserves traceability while reducing prompt noise\n', '- prompt construction is easier to debug when evidence boundaries stay visible\n', '- even a minimal answer layer should remain citation-aware\n', '\n', '

## Next steps

- Continue with [05_02_evidence_selection_and_topk_for_answering.ipynb](05_02_evidence_selection_and_topk_for_answering.ipynb) to isolate the answer-side evidence budget.
- Continue with [05_03_context_assembly_and_packing.ipynb](05_03_context_assembly_and_packing.ipynb) once you want to inspect the packed prompt boundary directly.
- Read [05-generation-overview.md](../../docs/05-generation-overview.md) for the chapter-level map of generation decisions.
